### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [ ]:
from pathlib import Path
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=Path(".env") if Path(".env").exists() else None)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in .env")
llm = init_chat_model(model="groq:llama-3.1-8b-instant")


In [ ]:
# Step 1: Load and embed the document
candidate_paths = [
    Path("10. Query Enhancement/2. langchain_crewai_dataset.txt"),
    Path("2. langchain_crewai_dataset.txt"),
]
file_path = next((path for path in candidate_paths if path.exists()), None)
if file_path is None:
    raise FileNotFoundError("Could not find 2. langchain_crewai_dataset.txt")
loader = TextLoader(str(file_path))
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [ ]:
import os


In [ ]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
Break the complex question into smaller retrieval-friendly sub-questions.
Return one question per line.

Question: {question}
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()


In [ ]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question": query})


In [ ]:
print(decomposition_question)


In [ ]:
# Step 4: QA chain per sub-question
answer_prompt = PromptTemplate.from_template("""
Answer the question from the context.

Context:
{context}

Question: {question}
""")
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def answer_subquestion(sub_q):
    docs = retriever.invoke(sub_q)
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": format_docs(docs), "question": sub_q})
    return {"question": sub_q, "answer": answer, "docs": docs}


In [ ]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("- ").strip() for q in sub_qs_text.splitlines() if q.strip()]
    sub_answers = [answer_subquestion(q) for q in sub_questions]
    synthesis_prompt = PromptTemplate.from_template("""
    Combine the following sub-answers into one final answer.
    
    Original question: {question}
    
    Sub-answers:
    {sub_answers}
    """)
    synthesis_chain = synthesis_prompt | llm | StrOutputParser()
    final_answer = synthesis_chain.invoke({
        "question": user_query,
        "sub_answers": "\n\n".join([f"Q: {x['question']}\nA: {x['answer']}" for x in sub_answers]),
    })
    return sub_questions, sub_answers, final_answer


In [ ]:
# Step 6: Run
sub_questions, sub_answers, final_answer = full_query_decomposition_rag_pipeline(query)
print(final_answer)
